# 04 · Summary statistics

From τ of several lines, `DeltaFieldBuilder` forms, per skewer, the overflux δ_j = F_j/⟨F_j⟩ − 1 of every line, the
products δ_A of every line subset and the total δ_tot = ∏F_j/⟨∏F_j⟩ − 1, which decomposes **exactly** into the
subset terms (notes `sec_flux.tex`, Sec. Overflux). `OverfluxStatistics1D` gives P1D and ξ1D (an exact discrete
Fourier pair; P_ab = (dv/N) F_a F_b^*, notes `subsec_P1D.tex`) of the total, of every line pair and of every order
|A| + |B|; `OverfluxStatistics3D` does the same on a grid of skewers. Runtime: ~30 s.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, TOL_BRIGHT as TOL, LABELS, FIGSIZES,
                                      add_panel_letters, plot_pipeline, plot_spectra, zoom_inset)
use_style()
%matplotlib inline

In [ ]:
plot_pipeline(('stats',), note="filled: the steps of this notebook"); plt.show()

In [ ]:
from skewerjax.core import PowerLawGRF, DeltaFieldBuilder
from skewerjax.core.stats import OverfluxStatistics1D, OverfluxStatistics3D, fit_mcdonald_a, fit_ma_model_profiled, mcdonald_factor, ma_factor
from skewerjax.utils.plotting import plot_orders_superimposed, plot_pk_recovery, hybrid_k_edges, logbin
N_sk, N_v, Lv = 400, 1000, 1000.0                         # skewers, pixels, velocity length [km/s]
dv, shift = Lv / N_v, 100                                  # the second line sits 100 km/s away
grf = PowerLawGRF(1.0, 2 * np.pi / 100, 0.5, L=Lv, N=N_v)
grf = PowerLawGRF(1.0 / float(grf.variance()), 2 * np.pi / 100, 0.5, L=Lv, N=N_v)     # sigma_delta = 1
delta = jax.vmap(grf.sample)(jax.random.split(jax.random.PRNGKey(7), N_sk))
tau = jnp.stack([jnp.exp(delta), 0.5 * jnp.exp(jnp.roll(delta, shift, axis=-1))])        # line 2: same gas, shifted, weaker
ov = DeltaFieldBuilder(tau, labels=["1", "2"], keep_fields=True).build()
print("subset keys:", ov.subset_keys, "| C_s of the first skewers:", np.round(np.asarray(ov.C_per_sample[:3, 0]), 4))
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True, layout="constrained")
vv = np.arange(N_v) * dv
for j, lab in enumerate(ov.labels):
    axs[0].plot(vv, ov.fields[j, 0], color=list(TOL.values())[j], label=f"$F_{lab}$"); axs[1].plot(vv, ov.delta_terms[lab][0], color=list(TOL.values())[j])
axs[1].plot(vv, ov.total[0], color="k", lw=1.2, label=r"$\delta_{\rm tot}$")
axs[0].set(ylabel="F", xlim=(0, 300)); axs[1].set(xlabel="v [km/s]", ylabel=r"$\delta$"); axs[0].legend(); axs[1].legend()
add_panel_letters(axs); fig.suptitle("Two lines of the same gas: the second is a shifted, weaker copy", fontsize=22); plt.show()

## 1 · P1D and ξ1D by order

Order n collects the terms (A, B) with |A| + |B| = n (order 0 is the normalisation term 𝒞). Their sum is the total to machine precision.

In [ ]:
st = OverfluxStatistics1D(ov, dv).compute_p1d().compute_xi1d()
fig, axs = plot_orders_superimposed(st, "P", figsize=(12, 8))
fig.suptitle("P1D: orders 2, 3, 4 add up to the total", fontsize=20)
fig, axs = plot_orders_superimposed(st, "xi", delta_v_list=[0.0, shift * dv], max_lag=300.0, figsize=(12, 8))
fig.suptitle(r"$\xi_{\rm 1D}$: the cross term peaks at the line separation (dashed)", fontsize=20); plt.show()

## 2 · Recovering an input spectrum, and the metal models

In the linear regime (σ_δ ≪ 1) the total P1D of a line plus a weaker copy at Δv is τ0²[1 + a² + 2a cos(kΔv)]P_in,
McDonald et al.'s Si III model. `fit_mcdonald_a` recovers a; Ma et al.'s model (`fit_ma_model_profiled`) adds
k-dependent amplitudes and contains McDonald's as a limit.

In [ ]:
a_in = 0.5
g_lin = PowerLawGRF(1e-4 * grf.A_s, grf.k_0, grf.n_s, L=Lv, N=N_v)
d = jax.vmap(g_lin.sample)(jax.random.split(jax.random.PRNGKey(8), N_sk))
st_l = OverfluxStatistics1D(DeltaFieldBuilder(jnp.stack([jnp.exp(d), a_in * jnp.exp(jnp.roll(d, shift, -1))])).build(), dv).compute_p1d()
k = np.asarray(st_l.k)[1:]; P_tot, P_11 = np.asarray(st_l.Pk_total)[1:], np.asarray(st_l.Pk_pair[("1", "1")])[1:]
fit = fit_mcdonald_a(k, P_tot, P_11, delta_v=shift * dv, k_range=(0.0, 1.0))
fma = fit_ma_model_profiled(k, P_tot, P_11, delta_v=shift * dv, k_range=(0.0, 1.0), run_global=False)
edges = hybrid_k_edges(2 * np.pi / Lv, float(k[-1]))
fig, axs = plot_pk_recovery(st_l.bin_p1d(edges), st_l.bin_p1d(edges, Pk=np.concatenate([[0], mcdonald_factor(k, a_in, shift * dv) * np.asarray(g_lin.power(k))]))[0],
                            label=r"$P_{\rm tot}$, 400 skewers", figsize=(13, 9))
axs[0].plot(k, mcdonald_factor(k, a_in, shift * dv) * np.asarray(g_lin.power(k)), color="k", label="McDonald, input a")
axs[0].legend(); fig.suptitle(f"Linear regime: P_tot follows McDonald's form; fitted a = {fit['a']:.4f} (input {a_in})", fontsize=20)
print(f"McDonald chi2 = {fit['chi2']:.3e} | Ma et al.: a_auto = {fma['a_auto']:.3f}, a_cross = {fma['a_cross']:.3f}, chi2 = {fma['chi2']:.3e}"); plt.show()

## 3D: P3D and ξ3D on a grid of skewers

The skewers of a regular transverse grid (row-major, N_sk = N_x N_y) give P3D(k_⊥, k_∥) and ξ3D(r_⊥, π); transverse
axes in cMpc, the line of sight in km/s (or anything else the user passes).

In [ ]:
g3 = PowerLawGRF(1.0, 1.0, -1.0, L=(40.0, 40.0, 40.0), N=(32, 32, 64))
d3 = g3.sample(jax.random.PRNGKey(3))
st3 = OverfluxStatistics3D(DeltaFieldBuilder(jnp.exp(0.3 * d3).reshape(1, 32 * 32, 64)).build(), 40.0 / 64, 40.0 / 32).compute_xi3d()
Pb, kperp, _, kpar, _ = st3.bin_p3d_kperp(5)
xb, rperp, _, pi, _ = st3.bin_xi3d_rperp(12)
from skewerjax.utils.plotting import tol_cmap
fig, axs = plt.subplots(1, 2, figsize=(20, 7), layout="constrained")
cm = tol_cmap("iridescent")
for i in range(len(kperp)):
    axs[0].loglog(np.asarray(kpar)[1:], np.asarray(Pb)[i, 1:], color=cm(0.25 + 0.7 * i / len(kperp)), label=f"$k_\perp$ = {float(kperp[i]):.2f}")
axs[0].set(xlabel=r"$k_\parallel$ [1/cMpc]", ylabel=r"$P_{\rm 3D}$", title=r"P3D in bins of $k_\perp$ [1/cMpc]"); axs[0].legend(fontsize=12)
lim = float(np.nanmax(np.abs(np.asarray(xb))))
im = axs[1].pcolormesh(np.asarray(pi), np.asarray(rperp), np.asarray(xb), cmap=tol_cmap("BuRd"), vmin=-lim, vmax=lim, shading="auto")
fig.colorbar(im, ax=axs[1], label=r"$\xi_{\rm 3D}$"); axs[1].set(xlabel=r"$\pi$ [cMpc]", ylabel=r"$r_\perp$ [cMpc]", title=r"$\xi_{\rm 3D}(r_\perp, \pi)$")
add_panel_letters(axs); fig.suptitle("3D statistics of a lognormal field on 32 × 32 skewers", fontsize=22); plt.show()

All the statistics kernels (`p1d_kernel`, `p3d_kernel`, `xi_from_power`, `binned_mean`) are jitted; recovery and exactness checks are in `scripts/validation/check_summary_statistics.py`.

## Summary

- The overflux of a product of lines decomposes exactly into subset terms. P1D and ξ1D split by pair and by order,
  and the orders add up to the total to rounding.
- In the linear regime the total P1D follows McDonald's form, and `fit_mcdonald_a` recovers the amplitude.
- Checks: `scripts/validation/check_summary_statistics.py`.